# Shape step walkthrough: one render, one photo

This notebook sends two images through the shape step of our pipeline and shows
a picture after every step:

1. find the brick (mask)
2. measure its size
3. cut out a grey 96 × 96 crop
4. optional: stretch the contrast
5. augmentation (only during training)
6. predict the part number (top 3)

Every step calls the real code in `src/legorec/shape/`. Two plots (the colour
distance in step 1 and the histogram in step 4) recompute an intermediate result
to show what happens inside a function.

**Before you run it**

- Install: `pip install -e ".[train,docs]" ipykernel`, then pick the `.venv`
  kernel (top right in VS Code).
- Checkpoints in `models/`: `shape.pt` (the team default, see the README) and,
  optional for the comparison, `shape_hki5v89j.pt` (the baseline).
- Two images in `data/raw/examples/` (this folder is not in Git): one render from
  the Kaggle dataset and one photo of a single brick. Or change the paths below.

Unknown words (render, top-1, …)? See the [glossary in the README](../README.md#glossary).

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from PIL import Image

from legorec.paths import MODELS_DIR, REPO_ROOT
from legorec.shape.augment import augment_batch, denormalize, normalize
from legorec.shape.model import ShapeClassifier
from legorec.shape.preprocess import brick_mask, preprocess_image

# Inputs: change these paths to try other images.
# The render is from the test split, so the models never saw it in training.
EXAMPLES = REPO_ROOT / "data" / "raw" / "examples"
IMAGE_PATHS = {
    "render": EXAMPLES / "3002 brick 2x3 011L.png",
    "photo": EXAMPLES / "3002_red_catalog_image.jpeg",
}
TRUE_PART = "3002"  # both images show a brick 2x3

MODEL_PATHS = {
    "team default": MODELS_DIR / "shape.pt",
    "baseline": MODELS_DIR / "shape_hki5v89j.pt",
}
FIGURES = REPO_ROOT / "docs" / "figures"  # generated, not in Git


def hide_ticks(ax):
    """No axis numbers, but keep the frame (a white photo stays visible)."""
    ax.set_xticks([])
    ax.set_yticks([])


def show(pictures, titles, size=3.5):
    """Draw images next to each other. Grey images are shown without automatic
    contrast: 0 is black, 255 is white."""
    _, axes = plt.subplots(1, len(pictures), figsize=(size * len(pictures), size))
    for ax, picture, title in zip(np.atleast_1d(axes), pictures, titles):
        ax.imshow(picture, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title)
        hide_ticks(ax)
    plt.show()

## Step 0: the two inputs

- **Render** (left): a computer-generated image from the Kaggle dataset,
  400 × 400 px. Grey brick, black background, random 3D pose. It is in the
  test split, so the models never saw it during training.
- **Photo** (right): here a catalog-style image of a red brick on a white
  background, not one of our own photos. Replace it with one of your own photos.

Both show the same part: **3002, brick 2×3**.

In [ ]:
images = {name: Image.open(path).convert("RGB") for name, path in IMAGE_PATHS.items()}

show(
    [np.asarray(image) for image in images.values()],
    [f"{name}: {image.width} x {image.height} px" for name, image in images.items()],
)

## Step 1: find the brick (`brick_mask`)

The border of the image shows the background. `brick_mask` takes the median
colour of the border and measures for every pixel how different its colour is
(left plot: dark = like the background, bright = very different). Pixels above
12 count as brick. Then it fills small gaps and keeps only the **largest**
region, so dust, crumbs or a line at the image edge do not count as brick
(right plot: white = brick).

In [ ]:
def colour_distance(rgb):
    """The first lines of brick_mask: how far each pixel's colour is from the
    border colour (in OpenCV's Lab colour space)."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    border = np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])
    return np.linalg.norm(lab - np.median(border, axis=0), axis=2)


masks = {}
for name, image in images.items():
    rgb = np.asarray(image)
    masks[name] = brick_mask(rgb)

    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4))
    heatmap = left.imshow(colour_distance(rgb), cmap="magma")
    fig.colorbar(heatmap, ax=left, label="colour distance")
    left.set_title(f"{name}: colour distance")
    right.imshow(masks[name], cmap="gray")
    right.set_title(f"{name}: mask")
    hide_ticks(left)
    hide_ticks(right)
    plt.show()

## Step 2: measure the size

The size of a brick is the longer side of the box around its mask, in pixels.
Why measure it? Step 3 scales every brick to 96 × 96, and after that a brick
1×3 and a brick 1×4 look alike. Models trained with `size_feature: true` get
this number as an extra input.

Photo pixels are not render pixels: the photo is bigger and was taken from
another distance. That is why size-feature models need `real_scale` for photos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (name, image) in zip(axes, images.items()):
    rows, columns = np.nonzero(masks[name])
    top, left = rows.min(), columns.min()
    height = rows.max() - top + 1
    width = columns.max() - left + 1
    _, size_px = preprocess_image(image)  # the second result is the measured size

    ax.imshow(image)
    box = Rectangle((left, top), width, height, fill=False, color="lime", linewidth=2)
    ax.add_patch(box)
    ax.set_title(f"{name}: longest side {size_px:.0f} px")
    hide_ticks(ax)
plt.show()

## Step 3: grey crop, 96 × 96 (what the baseline sees)

The image becomes grey: the shape classifier only looks at the shape, colour is
a separate step of the pipeline. Then a square around the brick (plus 8 %
margin) is cut out and scaled to 96 × 96 pixels.

Look at the background: black in the render, white in the photo. The renders
have a black background, so the baseline never saw a white one in training.
That is one part of the render-to-real gap.

In [ ]:
crops = {name: preprocess_image(image)[0] for name, image in images.items()}

show(list(crops.values()), [f"{name}: 96 x 96" for name in crops])

## Step 4 (option): contrast stretch

In grey, a red brick is almost black and a yellow one almost white, while the
renders are all mid-grey. With `contrast: stretch`, the grey values inside the
brick are spread over the full range 0..255 (the 2nd percentile becomes 0, the
98th becomes 255), and the background becomes one fixed grey (128).
The team default model was trained this way.

In [ ]:
stretched = {
    name: preprocess_image(image, contrast="stretch")[0]
    for name, image in images.items()
}
show(list(stretched.values()), [f"{name}: stretched" for name in stretched])

# Grey values of the brick: before (full image, inside the mask) and
# after (stretched crop without the grey background 128)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (name, image) in zip(axes, images.items()):
    before = np.asarray(image.convert("L"))[masks[name]]
    after = stretched[name][stretched[name] != 128]
    ax.hist(before, bins=64, range=(0, 255), density=True, alpha=0.6, label="before")
    ax.hist(after, bins=64, range=(0, 255), density=True, alpha=0.6, label="after")
    ax.set_title(f"{name}: grey values of the brick")
    ax.set_xlabel("grey value")
    ax.set_yticks([])
    ax.legend()
plt.show()

## Step 5: augmentation (only during training)

In training, every batch is changed at random: turned by any angle, zoomed and
shifted a little, made brighter or darker, with more or less contrast. So the
model has to learn the brick, not one exact picture. Below are eight random
versions of the stretched render crop: this is what training batches look like
for the team default model. Predictions (step 6) use the crop without changes.

In [ ]:
torch.manual_seed(0)  # the same random versions every time you run this cell

crop = torch.tensor(stretched["render"])  # (96, 96), grey values 0..255
batch = normalize(crop.repeat(8, 1, 1, 1))  # 8 copies, shape (8, 1, 96, 96)
augmented = denormalize(augment_batch(batch)) * 255  # back to grey values

show([picture[0].numpy() for picture in augmented], [""] * 8, size=1.8)

## Step 6: prediction, top 3

`ShapeClassifier.predict_images` runs steps 1–4 with the settings stored in the
checkpoint (for example `contrast`) and returns the three most likely parts.
We compare two models from the [experiment log](../docs/experiments.md):

- **team default** (`models/shape.pt`, at the moment run `56b289he`): trained
  with contrast stretch
- **baseline** (`models/shape_hki5v89j.pt`): trained without it

The green bar is the right part (3002). Both checkpoints were trained with the
older preprocessing `mask2`. For bricks that are not close to the image edge,
the current code gives the same crop.

In [ ]:
classifiers = {
    name: ShapeClassifier(path) for name, path in MODEL_PATHS.items() if path.exists()
}
names = next(iter(classifiers.values())).names  # part number -> readable name


def plot_top3(ax, guesses, title):
    """Horizontal bars for [(part, probability), ...]; the right part is green."""
    guesses = guesses[::-1]  # matplotlib draws the first bar at the bottom
    labels = [names[part] for part, _ in guesses]
    probabilities = [probability for _, probability in guesses]
    colours = ["tab:green" if part == TRUE_PART else "tab:gray" for part, _ in guesses]
    bars = ax.barh(labels, probabilities, color=colours)
    ax.bar_label(bars, labels=[f"{p:.0%}" for p in probabilities], padding=3)
    ax.set_xlim(0, 1.2)
    ax.set_xticks([])
    ax.set_title(title)


top3 = {}  # (model, image) -> [(part, probability), ...]
for model_name, classifier in classifiers.items():
    for image_name, image in images.items():
        results, _, _ = classifier.predict_images([image], k=3)
        top3[model_name, image_name] = results[0]

fig, axes = plt.subplots(
    len(images),
    len(classifiers),
    figsize=(6 * len(classifiers), 2.5 * len(images)),
    squeeze=False,
)
for row, image_name in enumerate(images):
    for column, model_name in enumerate(classifiers):
        ax = axes[row, column]
        plot_top3(ax, top3[model_name, image_name], f"{model_name}: {image_name}")
plt.tight_layout()
plt.show()

## What we see

With the example images above (2026-11-09):

- **Render:** both models find 3002, but only with about 67 % confidence. On
  renders the models are underconfident (experiment log, 2026-10-23, finding 6).
- **Photo:** both models answer 3001, brick 2×4, and they are very sure. This is
  the most common error type in the experiment log: the right kind of part in
  the wrong length. After step 3 the brick has lost its real size, and the photo
  also differs from the renders in background, colour, light and camera angle.

One example is not a measurement. Numbers on many images are in the
[experiment log](../docs/experiments.md).

**Try it yourself**

- The other renders in `data/raw/examples/` (poses `034L` and `049L`): there the
  two models disagree.
- One of your own photos: one brick, plain background, no strong shadow.
- A size-feature model: `ShapeClassifier(path, real_scale=...)`; the docstring of
  `scripts/predict_shape.py` explains a rough calibration.

## Figure for the slides

All steps in one picture, for the team default model. It is saved as
`docs/figures/shape_walkthrough.png` (generated, not in Git).

In [ ]:
default_model = classifiers["team default"]
column_titles = ["input", "1 · find the brick", "3–4 · model input", "6 · top 3"]

with plt.rc_context({"font.size": 14}):
    fig, axes = plt.subplots(
        len(images),
        4,
        figsize=(16, 3.8 * len(images)),
        width_ratios=[1, 1, 1, 1.4],
        squeeze=False,
        layout="constrained",
    )
    for row, (name, image) in enumerate(images.items()):
        model_input, _ = preprocess_image(image, contrast=default_model.contrast)
        pictures = [np.asarray(image), masks[name] * 255, model_input]
        for column, picture in enumerate(pictures):
            axes[row, column].imshow(picture, cmap="gray", vmin=0, vmax=255)
            hide_ticks(axes[row, column])
        axes[row, 0].set_ylabel(name, fontsize=18)
        plot_top3(axes[row, 3], top3["team default", name], title="")
    for ax, title in zip(axes[0], column_titles):
        ax.set_title(title)

FIGURES.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES / "shape_walkthrough.png", dpi=200, bbox_inches="tight")
plt.show()